# RFM clustering reusable template

**Short name (GitHub):** `CustSeg`

Clone this notebook onto another invoice ledger (another banner, year, or currency). Keep the grain honest: one RFM row per customer, log1p before scale, name bins from original-unit medians.

Replace the paths and column names in section 1. Everything downstream is generic.



## Contract

| Piece | Rule |
|-------|------|
| Input grain | one line item with customer id, timestamp, qty, unit price |
| Output grain | one row per customer + Cluster + Segment |
| Snapshot | max timestamp + 1 day (or a business month-end) |
| Frequency | distinct invoices / orders, never SKU count |
| Transforms | log1p → z-score; keep \(\mu,\sigma\) next to the model |
| k | elbow + silhouette + **ops capacity** (how many treatments can you fund?) |
| Naming | medians in original units; never the integer that K-Means happened to emit |



In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
import CustSeg as cs

# --- edit these ---
PATH = "data/online_retail.csv"
CUSTOMER = "CustomerID"
INVOICE = "InvoiceNo"
WHEN = "InvoiceDate"
QTY = "Quantity"
PRICE = "UnitPrice"
DATE_FMT = "%d.%m.%Y %H:%M"
K = 4
RANDOM_STATE = 42


In [ ]:
raw = pd.read_csv(PATH)
sales = raw.dropna(subset=[CUSTOMER]).copy()
sales = sales[sales[QTY] > 0]
sales = sales[sales[PRICE] > 0]
sales["TotalSum"] = sales[QTY] * sales[PRICE]
sales[WHEN] = pd.to_datetime(sales[WHEN], format=DATE_FMT)
sales[CUSTOMER] = sales[CUSTOMER].astype(int)

snap = sales[WHEN].max() + pd.Timedelta(days=1)
rfm = (
    sales.groupby(CUSTOMER)
    .agg(
        Recency=(WHEN, lambda x: (snap - x.max()).days),
        Frequency=(INVOICE, "nunique"),
        Monetary=("TotalSum", "sum"),
    )
    .reset_index()
)
print(len(rfm), "customers |", round(rfm["Monetary"].sum(), 2), "revenue | snapshot", snap)


In [ ]:
rfm_log = rfm[["Recency", "Frequency", "Monetary"]].apply(np.log1p)
scaler = StandardScaler()
X = scaler.fit_transform(rfm_log)

ks = list(range(1, 11))
inertias = [KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10).fit(X).inertia_ for k in ks]
print("inertia", [round(v, 1) for v in inertias])

model = KMeans(n_clusters=K, random_state=RANDOM_STATE, n_init=10).fit(X)
rfm["Cluster"] = model.labels_
print(rfm.groupby("Cluster")[["Recency", "Frequency", "Monetary"]].median().round(1))

Z = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(X)
fig, ax = plt.subplots()
ax.scatter(Z[:, 0], Z[:, 1], c=rfm["Cluster"], s=14, alpha=0.7, cmap="tab10")
ax.set_title(f"PCA view, k={K}")
plt.show()


## How to adapt

1. Confirm date format. Pandas ≥2 raises on mixed strings — set `DATE_FMT` or pass `dayfirst=True`.
2. If the ledger already has an `OrderID`, use that for Frequency.
3. If Monetary can be zero (free samples only), `log1p` still works; decide whether those customers belong in the book.
4. Recompute names every time you change k or the window. Do not hard-code `Cluster==2 → Champions` in production without a mapping table.
5. Persist `scaler.mean_`, `scaler.scale_`, `model.cluster_centers_`, and the name map together.

